## Part 1

In [39]:
pip install mysql-connector-python pandas

Note: you may need to restart the kernel to use updated packages.


In [40]:
import mysql.connector
 
mydb = mysql.connector.connect(
  host="localhost",
  user="root",
  password="qwertyuiop", #REPLACE THIS WITH YOUR PASSWORD 
  database = "data1050f26" #connecting to testDatabase
)
 
print(mydb)
 
if mydb.is_connected():
    print("CONNECTION SUCCESSFUL")


CONNECTION SUCCESSFUL


In [41]:
mycursor = mydb.cursor()
mycursor.execute("SHOW DATABASES")

for x in mycursor:
  print(x)

mycursor.execute("SHOW TABLES")

for x in mycursor:
  print(x)

('data1050f26',)
('information_schema',)
('mysql',)
('performance_schema',)
('sys',)
('advisor',)
('classroom',)
('course',)
('department',)
('instructor',)
('prereq',)
('student',)


In [42]:
#create time_slot table
mycursor = mydb.cursor()
mycursor.execute("""
CREATE TABLE data1050f26.time_slot (
    time_slot_id INT PRIMARY KEY,
    day VARCHAR(128) NOT NULL,
    start_time TIME NOT NULL,
    end_time TIME NOT NULL,
    CONSTRAINT uq_day_start UNIQUE (day, start_time),
    CONSTRAINT chk_day CHECK (day IN ('Monday','Tuesday','Wednesday','Thursday','Friday')),
    CONSTRAINT chk_time CHECK (end_time > start_time)
)
""")

In [43]:
#create a section table
mycursor.execute("""
CREATE TABLE data1050f26.section (
    course_id VARCHAR(128),
    sec_id VARCHAR(128),
    semester VARCHAR(16),
    year INT,
    building VARCHAR(128),
    room_no VARCHAR(128),
    time_slot_id INT,
    PRIMARY KEY (course_id, sec_id, semester, year),
    CONSTRAINT uq_sec_room_time UNIQUE (semester, year, building, room_no, time_slot_id),
    CONSTRAINT chk_sec_semester CHECK (semester IN ('Fall','Winter','Spring','Summer')),
    CONSTRAINT chk_sec_year CHECK (year > 1701 AND year < 2100),
    FOREIGN KEY (course_id) REFERENCES data1050f26.course(course_id),
    FOREIGN KEY (building, room_no) REFERENCES data1050f26.classroom(building, room_no),
    FOREIGN KEY (time_slot_id) REFERENCES data1050f26.time_slot(time_slot_id)
)
""")

In [44]:
#create a takes table
mycursor.execute("""
CREATE TABLE data1050f26.takes (
    ID VARCHAR(128),
    course_id VARCHAR(128),
    sec_id VARCHAR(128),
    semester VARCHAR(16),
    year INT,
    grade VARCHAR(2),
    PRIMARY KEY (ID, course_id, sec_id, semester, year),
    CONSTRAINT chk_takes_grade CHECK (grade IN
        ('A','B','C', 'NC', 'INC','S/NC')), #based on Bown's grading system
    FOREIGN KEY (ID) REFERENCES data1050f26.student(ID),
    FOREIGN KEY (course_id, sec_id, semester, year)
        REFERENCES data1050f26.section(course_id, sec_id, semester, year)
)
""")

In [45]:
#create a teaches table
mycursor.execute("""
CREATE TABLE data1050f26.teaches (
    ID VARCHAR(128),
    course_id VARCHAR(128),
    sec_id VARCHAR(128),
    semester VARCHAR(16),
    year INT,
    PRIMARY KEY (ID, course_id, sec_id, semester, year),
    FOREIGN KEY (ID) REFERENCES data1050f26.instructor(ID),
    FOREIGN KEY (course_id, sec_id, semester, year)
        REFERENCES data1050f26.section(course_id, sec_id, semester, year)
)
""")
mydb.commit()

In [46]:
mycursor.execute("SHOW TABLES")

for x in mycursor:
  print(x)


('advisor',)
('classroom',)
('course',)
('department',)
('instructor',)
('prereq',)
('section',)
('student',)
('takes',)
('teaches',)
('time_slot',)


## Part 2

In [47]:
import pandas as pd

In [48]:
pip install fsspec

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# populate data into time_slot table
data = pd.read_csv("time_slot.csv")
for i,row in data.iterrows():
            sql = "INSERT INTO time_slot VALUES (%s,%s,%s,%s)"
            mycursor.execute(sql, tuple(row))
            print("Record inserted")
            mydb.commit()

Record inserted
Record inserted
Record inserted
Record inserted
Record inserted
Record inserted
Record inserted
Record inserted
Record inserted
Record inserted
Record inserted
Record inserted
Record inserted
Record inserted
Record inserted
Record inserted
Record inserted
Record inserted


In [52]:
mycursor = mydb.cursor()
mycursor.execute("SELECT COUNT(*) FROM time_slot")
n_rows = mycursor.fetchone()[0]
print("Number of rows:", n_rows)


query = """
SELECT time_slot_id, day,
TIME_FORMAT(start_time, '%H:%i') AS start_time,
TIME_FORMAT(end_time,   '%H:%i') AS end_time
FROM time_slot
ORDER BY time_slot_id
LIMIT 5"""
mycursor.execute(query)
for x in mycursor:
    print(x)

Number of rows: 18
(1, 'Monday', '09:00', '11:00')
(2, 'Monday', '11:00', '13:00')
(3, 'Monday', '13:00', '15:00')
(4, 'Monday', '15:00', '17:00')
(5, 'Tuesday', '09:00', '11:00')


In [ ]:
# populate data into section table
data = pd.read_csv("section.csv", dtype=str)
rows = [
    tuple(None if pd.isna(v) else v for v in r)
    for r in data.itertuples(index=False)
]

query = "INSERT INTO section VALUES (%s,%s,%s,%s,%s,%s,%s)"
mycursor.executemany(query, rows)
mydb.commit()
print(mycursor.rowcount, "records inserted")

14 records inserted


In [ ]:
mycursor = mydb.cursor()
mycursor.execute("SELECT COUNT(*) FROM section")
n_rows = mycursor.fetchone()[0]
print("Number of rows:", n_rows)

query = """
SELECT * 
FROM section
LIMIT 5"""
mycursor.execute(query)
for x in mycursor:
    print(x)

Number of rows: 14
('Data1050', '1', 'Fall', 2022, 'CIT', '404', 3)
('Data1030', '1', 'Fall', 2022, 'Franklin', '212', 6)
('Data2050', '1', 'Fall', 2023, None, None, None)
('CSCI1270', '2', 'Fall', 2023, None, None, 6)
('Psyc2134', '1', 'Fall', 2023, 'Angell', '405', 6)


In [ ]:
# populate data into takes table
data = pd.read_csv("takes.csv", dtype=str)
rows = [
    tuple(None if pd.isna(v) else v for v in r)
    for r in data.itertuples(index=False)
]

query = "INSERT INTO takes VALUES (%s,%s,%s,%s,%s,%s)"
mycursor.executemany(query, rows)
mydb.commit()
print(mycursor.rowcount, "records inserted")

23 records inserted


In [62]:
mycursor = mydb.cursor()
mycursor.execute("SELECT COUNT(*) FROM takes")
n_rows = mycursor.fetchone()[0]
print("Number of rows:", n_rows)

query = """
SELECT * 
FROM takes
LIMIT 5"""
mycursor.execute(query)
for x in mycursor:
    print(x)

Number of rows: 23
('1122', 'CSCI1270', '2', 'Fall', 2023, None)
('1122', 'Data1050', '1', 'Fall', 2023, None)
('1238', 'CSCI1270', '1', 'Fall', 2023, None)
('1238', 'Data1030', '1', 'Fall', 2023, None)
('1238', 'Data1050', '1', 'Fall', 2022, 'A')


In [ ]:
# populate data into teaches table
data = pd.read_csv("teaches.csv", dtype=str)
rows = [
    tuple(None if pd.isna(v) else v for v in r)
    for r in data.itertuples(index=False)
]

query = "INSERT INTO teaches VALUES (%s,%s,%s,%s,%s)"
mycursor.executemany(query, rows)
mydb.commit()
print(mycursor.rowcount, "records inserted")

11 records inserted


In [65]:
mycursor = mydb.cursor()
mycursor.execute("SELECT COUNT(*) FROM teaches")
n_rows = mycursor.fetchone()[0]
print("Number of rows:", n_rows)

query = """
SELECT * 
FROM teaches
LIMIT 5"""
mycursor.execute(query)
for x in mycursor:
    print(x)

Number of rows: 11
('113', 'APMA1650', '1', 'Spring', 2023)
('112', 'CSCI1270', '1', 'Fall', 2023)
('118', 'CSCI1270', '2', 'Fall', 2023)
('117', 'Data1030', '1', 'Fall', 2023)
('117', 'Data1050', '1', 'Fall', 2022)


## Part 3

In [69]:
mycursor = mydb.cursor()
query = """
SELECT s.ID, s.name 
FROM student s
JOIN takes t ON s.ID = t.ID
WHERE s.dept_name = 'DSI' and 
t.course_id = 'Data1050' and 
t.semester = 'Fall' and 
t.year = 2023"""
mycursor.execute(query)
for x in mycursor:
    print(x)

('1845', 'Jane Chen')
('2142', 'Sabrina Zhou')
